In [ ]:
Save this file as Chinook_Sqlite.sql
Run sqlite3 Chinook.db
Run .read Chinook_Sqlite.sql
Test SELECT * FROM Artist LIMIT 10;

In [5]:
db_path = r'D:\projects\GENAI\llm\data\Chinook.db'

In [6]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri(f"sqlite:///{db_path}")
print(db.dialect)
print(db.get_usable_table_names())
print(db.run("SELECT * FROM Artist LIMIT 10;"))

sqlite
['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]


In [28]:
print(db.get_usable_table_names())

['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [2]:
from dotenv import load_dotenv
import os
load_dotenv()


True

In [7]:
from dotenv import load_dotenv
import os
load_dotenv()
from langchain_openai import ChatOpenAI
model=ChatOpenAI(model=os.getenv('model_name'),
                 api_key=os.getenv('token'))

In [8]:
table_names="\n".join(db.get_usable_table_names())
table_names

'Album\nArtist\nCustomer\nEmployee\nGenre\nInvoice\nInvoiceLine\nMediaType\nPlaylist\nPlaylistTrack\nTrack'

In [20]:
from langchain_core.prompts import ChatPromptTemplate
system=system = f"""Return the  SQL Query that MIGHT be relevant to the user question. \
The tables are:

{table_names}

Remember to include ALL POTENTIALLY sql query, even if you're not sure that they're needed."""

In [21]:

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{input}"),
    ]
)

In [22]:
from langchain_core.pydantic_v1 import BaseModel, Field
class Table(BaseModel):
    """Table in SQL database."""

    convert_into_sql_query: str = Field(description="convert into sql query")

In [23]:
llm_with_tools = model.with_structured_output(Table)

In [24]:
chain=prompt|llm_with_tools

In [36]:
chain_query=chain.invoke({"input": "What are all the distinct genres of Alanis Morissette songs"})
chain_query

Table(convert_into_sql_query="SELECT DISTINCT Genre.Name FROM Genre INNER JOIN Track ON Genre.GenreId = Track.GenreId INNER JOIN Album ON Track.AlbumId = Album.AlbumId INNER JOIN Artist ON Album.ArtistId = Artist.ArtistId WHERE Artist.Name = 'Alanis Morissette'")

In [37]:
query=chain_query.convert_into_sql_query
query

"SELECT DISTINCT Genre.Name FROM Genre INNER JOIN Track ON Genre.GenreId = Track.GenreId INNER JOIN Album ON Track.AlbumId = Album.AlbumId INNER JOIN Artist ON Album.ArtistId = Artist.ArtistId WHERE Artist.Name = 'Alanis Morissette'"

In [38]:
db.run(query)

"[('Rock',)]"